# La API en un entorno empresarial: un nivel por decisión de negocio

`api/main.py` sirve un modelo LightGBM independiente por nivel de la jerarquía M5
(ver `config/levels.py`). Este notebook se conecta a la API **ya levantada con
Docker** (`make docker-api`, puerto 8000) y simula, para tres niveles, cómo un
sistema externo pediría una predicción y tomaría una decisión simple con ella.

Cada escenario toma una fila real del *test* de backtest de ese nivel (mismo
enfoque que `scripts/test_api.py`) y la manda a `/predict/{level_id}` -- como es
una fila de test, también tenemos el real observado para ver el error.

In [1]:
import joblib
import pandas as pd
import requests
from loguru import logger

from api.registry import artifact_grain, resolve_artifact_path
from src.data.dataset import reconstruct_test_data

pd.set_option("display.max_columns", None)

BASE_URL = "http://localhost:8000"

try:
    requests.get(f"{BASE_URL}/health", timeout=2).raise_for_status()
except requests.exceptions.ConnectionError:
    raise RuntimeError("La API no responde en :8000 -- levantala con `make docker-api`")

logger.success("API OK en {}", BASE_URL)

2026-09-16 21:58:33.630 | SUCCESS  | __main__:<module>:18 - API OK en http://localhost:8000


## Helper: predecir una serie contra la API

Carga el artifact del nivel (misma resolución que usa la API), busca la fila de
`series_id` en la última fecha de test, y llama a `/predict/{level_id}` por HTTP.

In [2]:
def predict(level_id: int, series_id: str, grain: str = "daily") -> dict:
    artifact = joblib.load(resolve_artifact_path(level_id, "sales", grain=grain))
    X_test, _, _, test = reconstruct_test_data(artifact)

    last_date = test["date"].max()
    idx = test.index[(test["series_id"] == series_id) & (test["date"] == last_date)][0]
    row = X_test.loc[idx]

    features = {
        col: (None if pd.isna(row[col]) else
              str(row[col]) if col in artifact["categorical_features"] else float(row[col]))
        for col in artifact["features"]
    }
    resp = requests.post(
        f"{BASE_URL}/predict/{level_id}?grain={artifact_grain(artifact)}",
        json={"features": features},
    )
    resp.raise_for_status()

    pred = resp.json()["prediction"]
    real = float(test.loc[idx, "sales"])
    return {"series_id": series_id, "date": last_date.date(), "prediccion": round(pred, 1), "real": real,
            "error_pct": round(100 * (pred - real) / real, 1)}

## Escenario 1 -- Nivel 1 (total): guidance financiero

Finanzas quiere una proyección de ventas de la cadena completa antes del cierre.

In [3]:
predict(1, "TOTAL")

2026-09-16 21:58:35.264 | INFO     | src.data.dataset:load_data:58 - level_01_daily_total | target=sales | 1,941 filas | 191 features (9 categóricas)


2026-09-16 21:58:35.280 | INFO     | src.data.temporal_split:log_summary:41 - Train : 2011-01-29 to 2015-04-26 (1,549 days, 1,549 rows)


2026-09-16 21:58:35.281 | INFO     | src.data.temporal_split:log_summary:42 - Valid : 2015-04-27 to 2016-04-24 (364 days, 364 rows)


2026-09-16 21:58:35.284 | INFO     | src.data.temporal_split:log_summary:43 - Test  : 2016-04-25 to 2016-05-22 (28 days, 28 rows)


2026-09-16 21:58:35.493 | INFO     | src.data.dataset:split_data:119 - Features: 12 (1 categóricas)


{'series_id': 'TOTAL',
 'date': datetime.date(2016, 5, 22),
 'prediccion': 47287.3,
 'real': 54338.0,
 'error_pct': -13.0}

## Escenario 2 -- Nivel 3 (categoría): plan de compras con colchón de seguridad

Category management pide la orden de compra: pronóstico + `SAFETY_STOCK` para no
quedarse corto ante un pico no capturado por el modelo.

In [4]:
SAFETY_STOCK = 1.15

df_categoria = pd.DataFrame([predict(3, cat) for cat in ["FOODS", "HOBBIES", "HOUSEHOLD"]])
df_categoria["orden_sugerida"] = (df_categoria["prediccion"] * SAFETY_STOCK).round(0)
df_categoria

2026-09-16 21:58:35.748 | INFO     | src.data.dataset:load_data:58 - level_03_daily_cat | target=sales | 5,823 filas | 191 features (9 categóricas)


2026-09-16 21:58:35.765 | INFO     | src.data.temporal_split:log_summary:41 - Train : 2011-01-29 to 2015-04-26 (1,549 days, 4,647 rows)


2026-09-16 21:58:35.767 | INFO     | src.data.temporal_split:log_summary:42 - Valid : 2015-04-27 to 2016-04-24 (364 days, 1,092 rows)


2026-09-16 21:58:35.769 | INFO     | src.data.temporal_split:log_summary:43 - Test  : 2016-04-25 to 2016-05-22 (28 days, 84 rows)


2026-09-16 21:58:36.064 | INFO     | src.data.dataset:split_data:119 - Features: 57 (2 categóricas)


...

2026-09-16 21:58:36.899 | INFO     | src.data.temporal_split:log_summary:41 - Train : 2011-01-29 to 2015-04-26 (1,549 days, 4,647 rows)


2026-09-16 21:58:36.902 | INFO     | src.data.temporal_split:log_summary:42 - Valid : 2015-04-27 to 2016-04-24 (364 days, 1,092 rows)


2026-09-16 21:58:36.904 | INFO     | src.data.temporal_split:log_summary:43 - Test  : 2016-04-25 to 2016-05-22 (28 days, 84 rows)


2026-09-16 21:58:37.231 | INFO     | src.data.dataset:split_data:119 - Features: 57 (2 categóricas)


,series_id,date,prediccion,real,error_pct,orden_sugerida
0,FOODS,2016-05-22,31279.2,35967.0,-13.0,35971.0
1,HOBBIES,2016-05-22,4886.0,5280.0,-7.5,5619.0
2,HOUSEHOLD,2016-05-22,13027.3,13091.0,-0.5,14981.0


## Escenario 3 -- Nivel 6 (tienda): ¿turno extra de caja?

Regla simple: si la predicción supera `UMBRAL_TURNO_EXTRA` unidades/día, se
refuerza personal.

In [5]:
UMBRAL_TURNO_EXTRA = 6_000

df_tienda = pd.DataFrame([predict(6, tienda) for tienda in ["CA_1_CA", "CA_2_CA", "CA_3_CA", "CA_4_CA"]])
df_tienda["turno_extra"] = df_tienda["prediccion"] > UMBRAL_TURNO_EXTRA
df_tienda

2026-09-16 21:58:37.645 | INFO     | src.data.dataset:load_data:58 - level_06_daily_store | target=sales | 19,410 filas | 191 features (9 categóricas)


2026-09-16 21:58:37.674 | INFO     | src.data.temporal_split:log_summary:41 - Train : 2011-01-29 to 2015-04-26 (1,549 days, 15,490 rows)


2026-09-16 21:58:37.676 | INFO     | src.data.temporal_split:log_summary:42 - Valid : 2015-04-27 to 2016-04-24 (364 days, 3,640 rows)


2026-09-16 21:58:37.677 | INFO     | src.data.temporal_split:log_summary:43 - Test  : 2016-04-25 to 2016-05-22 (28 days, 280 rows)


2026-09-16 21:58:37.910 | INFO     | src.data.dataset:split_data:119 - Features: 6 (1 categóricas)


...

2026-09-16 21:58:39.242 | INFO     | src.data.temporal_split:log_summary:41 - Train : 2011-01-29 to 2015-04-26 (1,549 days, 15,490 rows)


2026-09-16 21:58:39.243 | INFO     | src.data.temporal_split:log_summary:42 - Valid : 2015-04-27 to 2016-04-24 (364 days, 3,640 rows)


2026-09-16 21:58:39.245 | INFO     | src.data.temporal_split:log_summary:43 - Test  : 2016-04-25 to 2016-05-22 (28 days, 280 rows)


2026-09-16 21:58:39.477 | INFO     | src.data.dataset:split_data:119 - Features: 6 (1 categóricas)


,series_id,date,prediccion,real,error_pct,turno_extra
0,CA_1_CA,2016-05-22,5693.5,6289.0,-9.5,False
1,CA_2_CA,2016-05-22,5648.4,6614.0,-14.6,False
2,CA_3_CA,2016-05-22,6879.0,8144.0,-15.5,True
3,CA_4_CA,2016-05-22,2895.0,3597.0,-19.5,False
